# 09｜编队与动态跟随


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


本课使用 `drone1`、`drone2`、`drone3`。

![协同与智能能力](assets/advanced.svg)


## 用布局表达编队

`FormationLayout` 负责产生跟随槽位，`fleet.follow()` 负责创建关系，`FollowSession` 负责运行控制循环。


In [ ]:
import time
from fleetdock_runtime import FleetDock
from fleetdock import FormationLayout

FleetDock.init()
leader = FleetDock.devices.drone1
followers = [FleetDock.devices.drone2, FleetDock.devices.drone3]
fleet = FleetDock.core.fleet(leader, *followers, name="V 型教学编队")

print(fleet.connect())
print(
    FleetDock.core.plan(fleet)
    .for_drones()
    .wait_until_ready(timeout_s=90.0)
    .unlock()
    .takeoff(altitude=15.0)
    .run(step_timeout_s=120.0)
)


In [ ]:
follow = fleet.follow(
    leader=leader,
    followers=followers,
    layout=FormationLayout.v_shape(spacing_m=8.0, count=2),
)

follow.start()
time.sleep(10.0)
print("跟随状态：", follow.status())

follow.transition_to(FormationLayout.column(spacing_m=8.0, count=2))
time.sleep(10.0)
print("切换后状态：", follow.status())

follow.stop()


In [ ]:
print(
    FleetDock.core.plan(fleet)
    .for_drones()
    .return_to_launch()
    .run(step_timeout_s=120.0)
)
FleetDock.close()


## 重点

业务代码只选择 leader、followers 和布局。目标计算、成员关系、策略切换和控制循环由 FleetDock SDK 维护。
